# Update 1 · Evolución del notebook 04

**Original conservado:** `04_Rendimiento_y_eventos.ipynb`.

**Primera parte:** se conservan sus 31 celdas originales, con cinco motores y eventos NASA.

**Segunda parte — Laboratorio P04:** descarga verificada de tres periodos completos PQRS; comprobación y conversión CSV → Parquet dentro del notebook; análisis equivalente con pandas CPU, Polars CPU y Polars GPU; calentamiento y tres repeticiones.

El laboratorio es independiente: comenzar en L1. No requiere completar P02 ni dispone de resultados precalculados.


# P04 e I01. Rendimiento y traza finita

P04: clase 7, 23 de octubre; I01: clase 8, 24 de octubre de 2026.
Preparado por el PhD Esteban Hernández, CyberColombia.

[Enunciado P04](../../talleres/P04.md) · [I01](../../talleres/I01.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

El viernes el profesor demuestra; el sábado los estudiantes ejecutan y entregan en clase. Ejecutar las celdas en orden con el kernel del único `.venv`. Cada lectura, consulta y escritura queda visible. No se importan scripts del curso.

## 1. Preparar y localizar los datos
Desde la terminal en `kit/`, con `.venv` activo:
```bash
python 00_datos.py --descargar pqrs nasa
python 00_datos.py --verificar pqrs nasa
```
Primero ejecutar [02 Parquet y contratos](02_Parquet_y_contratos.ipynb): allí se leen los CSV, se convierten los tipos y se escriben el CSV proyectado, el Parquet y sus particiones. Aquí se leen esos productos; no se regeneran silenciosamente. Las muestras contienen 3.000 reportes; no permiten concluir sobre rendimiento nacional.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "pandas/P04"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
entrada = KIT / "salidas/pandas/P02"
archivo_parquet = entrada / "pqrs.parquet"
archivo_csv = entrada / "pqrs_proyeccion.csv"
particiones = entrada / "pqrs_particionado"
assert archivo_parquet.is_file(), "Ejecuta el notebook 02"
assert archivo_csv.is_file() and list(particiones.glob("**/*.parquet"))
control_entrada = json.loads((entrada / "control_formatos_pandas.json").read_text())
assert control_entrada["filas"] == 3000, "Este bloque mide las muestras de P02"
columnas = ["corte_archivo", "anio", "mes_num", "afec_cod_depto"]
# Importar antes de medir: excluimos el coste de importación de las bibliotecas.
import polars as pl
import duckdb
from time import perf_counter
import platform
mediciones = []
display(pd.DataFrame([{"python": platform.python_version(), "pandas": pd.__version__,
                       "polars": pl.__version__, "duckdb": duckdb.__version__,
                       "sistema": platform.platform()}]))


## 2. Comparar la misma consulta en cinco variantes
Agrupar **todos los años** por corte, año, mes y departamento, incluyendo claves nulas, y contar filas. Cada celda mide lectura + agregación + obtención del resultado en pandas. `%%time` informa CPU y **Wall time**; `perf_counter` guarda el tiempo transcurrido del bloque para el informe. Polars incluye `to_pandas`; DuckDB incluye la apertura/cierre de conexión.

La ordenación, normalización de tipos, comprobación y visualización se realizan después y quedan fuera del intervalo registrado. Ejecutar una vuelta de calentamiento, vaciar `mediciones` con la celda indicada y realizar tres vueltas, alternando el orden de los motores. No volver a ejecutar la configuración entre vueltas. La caché del sistema puede persistir; no llamar a esto una prueba con caché fría. Este ejercicio comparte el proceso del kernel y no mide RSS ni arranque de procesos.


### pandas: lectura inmediata y groupby


In [ ]:
%%time
inicio = perf_counter()
datos_pandas = pd.read_parquet(archivo_parquet, columns=columnas)
resultado_pandas = (datos_pandas.groupby(columnas, dropna=False)
    .size().reset_index(name="reportes"))
segundos = perf_counter() - inicio
mediciones.append({"motor": "pandas", "segundos": segundos})


### Polars: lectura diferida y ejecución explícita
`scan_parquet` prepara el plan; `collect` lo ejecuta. Medir solo la creación de `consulta` no mide el procesamiento.


In [ ]:
%%time
inicio = perf_counter()
consulta = (pl.scan_parquet(archivo_parquet)
    .group_by(columnas).agg(pl.len().alias("reportes")))
resultado_polars = consulta.collect(engine="streaming").to_pandas()
segundos = perf_counter() - inicio
mediciones.append({"motor": "polars", "segundos": segundos})


### DuckDB: SQL sobre Parquet
Las tres variantes siguientes usan exactamente el mismo SQL. El CSV se lee como texto para conservar los ceros de los códigos; por ello incluye un coste distinto de interpretación del formato.


In [ ]:
sql = """
SELECT corte_archivo, anio, mes_num, afec_cod_depto, COUNT(*) AS reportes
FROM reportes
GROUP BY corte_archivo, anio, mes_num, afec_cod_depto
"""


### duckdb


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(archivo_parquet)).create_view("reportes")
    resultado_duckdb = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb", "segundos": segundos})


### duckdb_particionado


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(particiones / "**/*.parquet"), hive_partitioning=True).create_view("reportes")
    resultado_duckdb_particionado = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_particionado", "segundos": segundos})


### duckdb_csv


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_csv(str(archivo_csv), header=True, all_varchar=True).create_view("reportes")
    resultado_duckdb_csv = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_csv", "segundos": segundos})


## 3. Verificar antes de comparar tiempos
El CSV devuelve claves numéricas como texto; armonizamos tipos fuera de la medición. No basta con comparar el total: cada grupo debe coincidir. Inspeccionar primero los tipos originales. Repetir esta comprobación después de cada vuelta.


In [ ]:
resultados = {"pandas": resultado_pandas, "polars": resultado_polars,
              "duckdb": resultado_duckdb, "duckdb_particionado": resultado_duckdb_particionado,
              "duckdb_csv": resultado_duckdb_csv}
comparables = {}
for motor, resultado in resultados.items():
    print(motor, resultado.dtypes.to_dict())
    normalizado = resultado.copy()
    for columna in ["anio", "mes_num", "reportes"]:
        normalizado[columna] = pd.to_numeric(normalizado[columna], errors="raise").astype("Int64")
    normalizado["corte_archivo"] = normalizado["corte_archivo"].astype("string")
    normalizado["afec_cod_depto"] = normalizado["afec_cod_depto"].astype("string").str.zfill(2)
    comparables[motor] = normalizado.sort_values(columnas).reset_index(drop=True)
    assert normalizado["reportes"].sum() == 3000
for motor, resultado in comparables.items():
    pd.testing.assert_frame_equal(comparables["pandas"], resultado)
display(comparables["pandas"])
print("Los cinco resultados coinciden grupo por grupo")


### Excluir el calentamiento
Ejecutar la siguiente celda **una vez**, tras verificar la primera vuelta. Después repetir únicamente las cinco celdas medidas y la comprobación anterior tres veces; alternar su orden. No ejecutar de nuevo esta celda al finalizar.


In [ ]:
mediciones = []


### Guardar las tres repeticiones
Ejecutar tras completar las tres vueltas. Este archivo documenta el alcance del kernel; no sustituye las mediciones de RSS en procesos independientes.


In [ ]:
tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == 15, "Faltan las tres vueltas de cinco motores después del calentamiento"
assert tiempos.groupby("motor").size().eq(3).all()
tiempos["repeticion"] = tiempos.groupby("motor").cumcount() + 1
tiempos["alcance"] = "kernel: lectura, agregación y resultado pandas; 3000 filas"
resumen = tiempos.groupby("motor")["segundos"].agg(["median", "min", "max"]).reset_index()
tiempos.to_csv(OUT / "benchmark_notebook.csv", index=False)
resumen.to_csv(OUT / "benchmark_notebook_resumen.csv", index=False)
comparables["pandas"].to_csv(OUT / "conteos_verificados.csv", index=False)
display(resumen)


## 4. Escala real y automatización, después de comprender las consultas
[P04](../../talleres/P04.md) conserva la extensión de cinco variantes en procesos independientes, dos hilos, calentamiento, tres repeticiones y sondeo RSS cada 10 ms. Revisar `trabajador` y `benchmark` en `kit/pqrs_talleres.py`: identificar en el código cada consulta que acabas de ejecutar antes de usar la automatización.

Los tiempos del CLI incluyen arranque y exportación; **no mezclar** esas mediciones con las del kernel. El CLI escribe `benchmark_pqrs.csv` y su resumen; el notebook usa otros nombres para distinguir el alcance. Registrar máquina, versiones e hilos. Las bibliotecas del kernel pueden usar distintos números de hilos: esta comparación describe la configuración local, no aísla el efecto del motor.

El volumen completo se descarga explícitamente según P04; reservar al menos 8 GB de disco más el entorno. Con 8 GB de RAM mantener pandas en muestras. Los completos concilian 2.444.766 reportes. No cambiar la ruta de este bloque de muestras a los completos sin revisar su preparación y memoria.

**Detenerse aquí en la clase 7.** Retomar eventos en la clase 8.

## 5. Leer las observaciones NASA
`json.load` lee el documento; `DataFrame` convierte el diccionario de parámetros en filas. Convertimos las fechas y separamos los faltantes (-999), sin inventar lluvia. La fecha diaria se usa como tiempo de evento por convención didáctica.


In [ ]:
with open(RAW / "nasa.json", encoding="utf-8") as archivo:
    nasa = json.load(archivo)
lluvia = pd.Series(nasa["properties"]["parameter"]["PRECTOTCORR"], name="lluvia_mm")
observaciones = lluvia.rename_axis("event_id").reset_index()
observaciones["fecha_evento"] = pd.to_datetime(observaciones["event_id"], format="%Y%m%d")
observaciones["lluvia_mm"] = pd.to_numeric(observaciones["lluvia_mm"], errors="raise").replace(-999, float("nan"))
assert observaciones["event_id"].is_unique
display(observaciones.head(12))


## 6. Construir el orden de entrega y unir los valores reales
Primero llegan los días 1–4 y 6–12; luego se reintenta el 6 y finalmente llega el 5. `merge(validate="many_to_one")` permite entregas repetidas pero exige una observación por fecha. Inspeccionar la tabla antes de aplicar reglas.


In [ ]:
dias = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({"event_id": [f"202501{dia:02d}" for dia in dias]})
entregas["orden_entrega"] = range(1, len(entregas) + 1)
entregas = entregas.merge(observaciones, on="event_id", how="left", validate="many_to_one", indicator=True)
assert entregas["_merge"].eq("both").all() and entregas["lluvia_mm"].notna().all()
display(entregas)


## 7. Recorrer la traza paso a paso
Antes de cada entrega, el límite es la fecha máxima observada menos tres días. Primero comprobamos si el identificador ya fue aceptado; después, si la fecha es anterior al límite. Una fecha igual al límite se admite. La lista `traza` conserva la decisión y el límite usado; `vistos` solo guarda aceptados. Esta regla finita no reproduce toda la semántica de Spark.


In [ ]:
vistos = set()
ultima_fecha = None
traza = []
for entrega in entregas.itertuples(index=False):
    limite = ultima_fecha - pd.Timedelta(days=3) if ultima_fecha is not None else None
    if entrega.event_id in vistos:
        estado = "duplicado"
    elif limite is not None and entrega.fecha_evento < limite:
        estado = "tardio"
    else:
        estado = "aceptado"
        vistos.add(entrega.event_id)
    traza.append({"orden_entrega": entrega.orden_entrega, "event_id": entrega.event_id,
                  "fecha_evento": entrega.fecha_evento, "lluvia_mm": entrega.lluvia_mm,
                  "watermark_antes": limite, "estado": estado})
    ultima_fecha = max(ultima_fecha, entrega.fecha_evento) if ultima_fecha is not None else entrega.fecha_evento
    print(entrega.orden_entrega, entrega.event_id, "límite:", limite, estado)
traza = pd.DataFrame(traza)


## 8. Conciliar y escribir las evidencias
Comparar entregas, fechas distintas y aceptados. Explicar qué observación falta en la suma aceptada y proponer una corrección antes de continuar con Spark en E11.


In [ ]:
aceptados = traza.loc[traza["estado"].eq("aceptado")]
control = {"entregas": len(traza), "registros_reales_distintos": int(traza["event_id"].nunique()),
           "aceptados": len(aceptados), "duplicados": int(traza["estado"].eq("duplicado").sum()),
           "tardios": int(traza["estado"].eq("tardio").sum()),
           "suma_aceptada_mm": float(aceptados["lluvia_mm"].sum()),
           "suma_fechas_unicas_mm": float(entregas.drop_duplicates("event_id")["lluvia_mm"].sum())}
assert (control["entregas"], control["registros_reales_distintos"], control["aceptados"], control["duplicados"], control["tardios"]) == (13, 12, 11, 1, 1)
assert abs(control["suma_aceptada_mm"] - 41.44) < 1e-8
traza.to_csv(OUT / "traza_eventos_nasa.csv", index=False)
with open(OUT / "control_eventos_intro.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
recuperada = pd.read_csv(OUT / "traza_eventos_nasa.csv", dtype={"event_id": "string"},
                         parse_dates=["fecha_evento", "watermark_antes"])
pd.testing.assert_frame_equal(traza.astype({"event_id": "string"}), recuperada)
display(control)
display(recuperada)


## Entrega
Adjuntar tiempos, resultados equivalentes, configuración y alcance de la medición; distinguir notebook y CLI. Entregar la traza y explicar el reintento y el atraso. Modificar una condición de consulta, anticipar el resultado y verificarlo en todos los motores. No sumar lluvia con PQRS ni fabricar tiempos de evento para reportes.


# Laboratorio P04 · Grandes datos PQRS: pandas, Polars y GPU

**Pregunta:** ¿cuánto cuesta preparar los datos y cuánto cuesta analizar los
mismos reportes con pandas CPU, Polars CPU y Polars GPU?

Trabajaremos con los CSV completos de **2023-II, 2024-I y 2024-II**:
716.697 + 781.601 + 946.468 = **2.444.766 reportes**, aproximadamente **1,82 GB**.
Son conteos del manifiesto docente, que se comprobarán al ejecutar.
No son las muestras de 1.000 filas y no se multiplican registros artificialmente.

Este bloque es independiente de la primera parte: comenzar en su configuración.
No necesita los Parquet de P02. Aquí se hace todo el recorrido:

**comprobar CSV → descargar faltantes → comprobar Parquet → convertir faltantes
→ validar los tres periodos → analizar → comprobar equivalencia → medir**.

Preparar antes el entorno `.venv-gpu` y su kernel según la
[guía NVIDIA](../../docs/polars-gpu.md). No se instalan paquetes desde el notebook.
La GPU Intel no participa en este backend CUDA. Se requiere NVIDIA compatible.
Este laboratorio se entrega sin ejecutar: no incluye tiempos ni resultados GPU inventados.


## L1. Configuración y contrato
Editar `REPOSITORIO` si no se detecta. `BLOQUE` limita el bloque de pandas,
no la RAM total del proceso. Ambas bibliotecas leen los mismos Parquet.
Los CSV originales se conservan, con sus 38 columnas, sus duplicados y sus vacíos.
El manifiesto contiene URL oficial, separador, filas, tamaño y SHA-256 por corte.


In [ ]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
from IPython.display import display
import csv, hashlib, json, os, platform, shutil, subprocess, sys
import urllib.request
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

REPOSITORIO = None  # Ejemplo WSL: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
PERIODOS = ['2023_II', '2024_I', '2024_II']
BLOQUE = 50_000
REPETICIONES = 3
DISPOSITIVO_CUDA = 0  # Índice CUDA, no el número de GPU del Administrador de tareas
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Configurar REPOSITORIO'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
KIT = REPOSITORIO / 'kit'
CSV_DIR = KIT / 'data/pqrs_completos'
OUT = KIT / 'salidas/pandas/P04_update1'
PARQUET_DIR = OUT / 'parquet'
for carpeta in [CSV_DIR, OUT, PARQUET_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)
manifiesto = json.loads((KIT / 'pqrs_fuentes.json').read_text(encoding='utf-8'))
por_periodo = {f['periodo']: f for f in manifiesto}
assert len(PERIODOS) >= 3 and len(set(PERIODOS)) == len(PERIODOS)
assert set(PERIODOS) <= set(por_periodo), 'Periodo no incluido en el manifiesto'
FUENTES = [por_periodo[p] for p in PERIODOS]
TOTAL_ESPERADO = sum(f['filas'] for f in FUENTES)
assert BLOQUE > 0 and REPETICIONES >= 3
CLAVES = ['corte_archivo', 'anio', 'mes_num', 'afec_cod_depto']
CONSULTAS = {'territorio_mes': CLAVES,
             'territorio_mes_entidad': CLAVES + ['ent_nombre']}
print('Intérprete:', sys.executable)
print('Versiones:', {'pandas': pd.__version__, 'polars': pl.__version__, 'pyarrow': pa.__version__})
display(pd.DataFrame([{k: f[k] for k in ['periodo', 'archivo', 'filas', 'bytes', 'separador', 'url']}
                      for f in FUENTES]))
print('Total esperado:', TOTAL_ESPERADO, 'reportes')
print('Libres en disco (GiB):', shutil.disk_usage(KIT).free / 2**30)


## L2. ¿Ya se descargaron los tres CSV?
Para cada periodo se comprueba existencia, tamaño, SHA-256 y cabecera.
**Si existe y coincide, se reutiliza. Si falta, se descarga. Si existe pero no
coincide, se detiene:** no se sobrescribe una fuente distinta sin revisarla.
La descarga es secuencial, por bloques de 1 MiB. No se carga el CSV en RAM.
Si falla la red, se puede reejecutar la celda; el archivo parcial no cuenta
como descarga completa. Un hash nuevo no se acepta automáticamente.

El tiempo de esta celda es preparación y **no entra en el benchmark**.


In [ ]:
def sha256_archivo(archivo):
    huella = hashlib.sha256()
    with archivo.open('rb') as stream:
        for bloque in iter(lambda: stream.read(1024 * 1024), b''):
            huella.update(bloque)
    return huella.hexdigest()

descargas = []
for fuente in FUENTES:
    archivo = CSV_DIR / fuente['archivo']
    estado = 'reutilizado'
    if not archivo.exists():
        print('Descargando:', fuente['periodo'], fuente['url'], flush=True)
        temporal = archivo.with_suffix('.csv.part')
        inicio = perf_counter()
        solicitud = urllib.request.Request(fuente['url'], headers={'User-Agent': 'Curso-BigData/1.0'})
        with urllib.request.urlopen(solicitud, timeout=60) as respuesta, temporal.open('wb') as destino:
            assert 'text/html' not in respuesta.headers.get('Content-Type', '').lower(), 'Se recibió HTML'
            recibidos = 0
            for bloque in iter(lambda: respuesta.read(1024 * 1024), b''):
                destino.write(bloque)
                recibidos += len(bloque)
                if recibidos > fuente['bytes'] * 1.5:
                    raise ValueError('Tamaño inesperado: revisar la fuente antes de continuar')
        if temporal.stat().st_size != fuente['bytes'] or sha256_archivo(temporal) != fuente['sha256']:
            raise ValueError(f'No coincide el corte docente; conservar y revisar {temporal}')
        temporal.replace(archivo)
        estado = 'descargado'
        print('Descarga (s):', round(perf_counter() - inicio, 2), flush=True)
    assert archivo.stat().st_size == fuente['bytes'], f'Tamaño distinto: {archivo}'
    assert sha256_archivo(archivo) == fuente['sha256'], f'SHA-256 distinto: {archivo}'
    with archivo.open(encoding=fuente.get('codificacion', 'utf-8-sig'), newline='') as stream:
        cabecera = next(csv.reader(stream, delimiter=fuente['separador']))
    assert cabecera == fuente['columnas'], f'Cabecera distinta: {archivo}'
    descargas.append({'periodo': fuente['periodo'], 'archivo': str(archivo),
                      'estado': estado, 'bytes': archivo.stat().st_size, 'sha256_correcto': True})
    print(fuente['periodo'], estado, 'VERIFICADO', flush=True)
assert len(descargas) == len(FUENTES)
display(pd.DataFrame(descargas))


## L3. Mirar los datos antes de convertir
Leer cinco filas de cada CSV usando **su propio separador**. Comparar `periodo`
(año informado en la fuente) con el corte semestral del archivo. No fabricar
el mes desde el nombre del corte. Los códigos se leen como texto para conservar
ceros iniciales; no se interpretan textos como `NA` automáticamente como nulos.


In [ ]:
for fuente in FUENTES:
    vista = pd.read_csv(CSV_DIR / fuente['archivo'], sep=fuente['separador'],
        encoding=fuente.get('codificacion', 'utf-8-sig'), dtype=str,
        keep_default_na=False, nrows=5)
    print('Periodo:', fuente['periodo'], 'Columnas:', len(vista.columns))
    display(vista[['OBJECTID', 'periodo', 'mes', 'afec_cod_depto', 'ent_nombre']])


## L4. ¿Ya están convertidos TODOS los periodos?
Cada CSV se transforma a un Parquet propio. Conservamos las **38 columnas
originales como texto** y añadimos `corte_archivo`, `anio` y `mes_num`.
Las conversiones numéricas inválidas quedan nulas: no se eliminan reportes.

Una conversión solo se considera válida si existen Parquet y control y coinciden
la fuente SHA-256, el esquema, el número de filas y la huella del Parquet.
Si falta o quedó incompleta, se vuelve a convertir ese periodo. Cada archivo se
publica después de comprobar su conteo. Se usan bloques pandas → Arrow → Parquet
con compresión Zstandard. **La conversión completa está aquí, no en un script externo.**

Se reutilizan exactamente estos productos para todos los motores del benchmark;
no se compara un CSV ancho con una tabla previamente reducida.


In [ ]:
conversiones = []
for fuente in FUENTES:
    periodo = fuente['periodo']
    parquet = PARQUET_DIR / f'{periodo}.parquet'
    control_path = PARQUET_DIR / f'{periodo}.control.json'
    esquema = pa.schema([(nombre, pa.string()) for nombre in fuente['columnas']] +
        [('corte_archivo', pa.string()), ('anio', pa.int64()), ('mes_num', pa.int64())])
    valido = False
    if parquet.is_file() and control_path.is_file():
        try:
            anterior = json.loads(control_path.read_text())
            pf = pq.ParquetFile(parquet)
            valido = (anterior['sha256_csv'] == fuente['sha256'] and
                anterior['filas'] == fuente['filas'] == pf.metadata.num_rows and
                pf.schema_arrow.equals(esquema, check_metadata=False) and
                anterior['sha256_parquet'] == sha256_archivo(parquet))
        except (OSError, ValueError, KeyError, pa.ArrowException):
            valido = False
    if valido:
        conversiones.append({**anterior, 'estado': 'reutilizado_verificado'})
        print(periodo, 'Parquet ya convertido y verificado', flush=True)
        continue
    print('Convirtiendo CSV → Parquet:', periodo, flush=True)
    inicio = perf_counter()
    temporal = parquet.with_suffix('.parquet.part')
    filas = 0
    with pq.ParquetWriter(temporal, esquema, compression='zstd') as escritor:
        lector = pd.read_csv(CSV_DIR / fuente['archivo'], sep=fuente['separador'],
            encoding=fuente.get('codificacion', 'utf-8-sig'), dtype=str,
            keep_default_na=False, chunksize=BLOQUE)
        for bloque in lector:
            assert list(bloque.columns) == fuente['columnas']
            bloque['corte_archivo'] = periodo
            bloque['anio'] = pd.to_numeric(bloque['periodo'], errors='coerce').astype('Int64')
            bloque['mes_num'] = pd.to_numeric(bloque['mes'], errors='coerce').astype('Int64')
            tabla = pa.Table.from_pandas(bloque, schema=esquema, preserve_index=False, safe=True)
            escritor.write_table(tabla)
            filas += len(bloque)
    segundos_conversion = perf_counter() - inicio
    assert filas == fuente['filas'], f'Conteo distinto en {periodo}: {filas}'
    assert pq.ParquetFile(temporal).metadata.num_rows == filas
    temporal.replace(parquet)
    control = {'periodo': periodo, 'archivo': str(parquet), 'filas': filas,
        'columnas': len(esquema), 'sha256_csv': fuente['sha256'],
        'sha256_parquet': sha256_archivo(parquet),
        'bytes_csv': fuente['bytes'], 'bytes_parquet': parquet.stat().st_size,
        'segundos_conversion': segundos_conversion, 'bloque_pandas': BLOQUE}
    control_path.write_text(json.dumps(control, ensure_ascii=False, indent=2))
    conversiones.append({**control, 'estado': 'convertido_verificado'})

assert len(conversiones) == len(FUENTES), 'No continuar: faltan periodos convertidos'
assert sum(c['filas'] for c in conversiones) == TOTAL_ESPERADO
ARCHIVOS_PARQUET = [PARQUET_DIR / f"{f['periodo']}.parquet" for f in FUENTES]
assert all(p.is_file() for p in ARCHIVOS_PARQUET)
display(pd.DataFrame(conversiones)[['periodo', 'estado', 'filas', 'columnas',
    'bytes_csv', 'bytes_parquet', 'segundos_conversion']])
print('TODOS los periodos descargados, convertidos y verificados. Se puede analizar.')


## L5. Preguntas de análisis y fronteras del experimento

1. **Volumen territorial:** ¿cuántos reportes hay por corte, año, mes y departamento?
2. **Entidad y territorio:** ¿cómo cambia la distribución al añadir la entidad?

La segunda consulta tiene potencialmente más grupos y exige más memoria de
agregación. Ninguna cuenta personas únicas ni elimina duplicados. Los códigos
vacíos forman un grupo y los años/meses nulos se conservan.

El benchmark mide **leer Parquet + agregar + devolver resultado a CPU**, para
las mismas columnas y todos los periodos. Excluye descarga, conversión,
creación del plan, ordenación de comprobación, validación y escritura de evidencia.
Pandas procesa lotes; Polars CPU usa streaming; Polars GPU usa cuDF. Se comparan
esas estrategias declaradas, no únicamente instrucciones de CPU frente a GPU.


## L6. Análisis con pandas: agregaciones parciales por lote
Solo se leen las columnas necesarias. Cada lote produce conteos por clave;
después se suman los conteos de los lotes. No se calcula un promedio de promedios.
Los grupos acumulados también consumen RAM: el tamaño del lote no garantiza
que cualquier agregación quepa en memoria.


In [ ]:
def analizar_pandas(claves):
    parciales = []
    for archivo in ARCHIVOS_PARQUET:
        for lote in pq.ParquetFile(archivo).iter_batches(batch_size=BLOQUE, columns=claves):
            datos = lote.to_pandas()
            for nombre in ['anio', 'mes_num']:
                datos[nombre] = pd.to_numeric(datos[nombre], errors='raise').astype('Int64')
            parcial = datos.groupby(claves, dropna=False).size().reset_index(name='reportes')
            parciales.append(parcial)
    acumulados = pd.concat(parciales, ignore_index=True)
    return acumulados.groupby(claves, dropna=False, as_index=False)['reportes'].sum()

referencias = {}
primeras = []
for nombre, claves in CONSULTAS.items():
    inicio = perf_counter()
    resultado = analizar_pandas(claves)
    primeras.append({'consulta': nombre, 'motor': 'pandas_cpu',
                     'segundos': perf_counter() - inicio})
    assert int(resultado['reportes'].sum()) == TOTAL_ESPERADO
    referencias[nombre] = resultado
    print(nombre, 'reportes:', resultado['reportes'].sum(), 'grupos:', len(resultado))
    display(resultado.head(10))


## L7. Análisis con Polars CPU: plan lazy y ejecución
`scan_parquet` prepara la lectura; el trabajo se ejecuta en `collect`.
No se cronometra solo la creación del plan. El resultado vuelve a pandas para
usar la misma representación final de comparación que los otros motores.


In [ ]:
consultas_polars = {
    nombre: (pl.scan_parquet(ARCHIVOS_PARQUET).select(claves)
        .group_by(claves).agg(pl.len().alias('reportes')))
    for nombre, claves in CONSULTAS.items()
}
print(consultas_polars['territorio_mes'].explain())

def comparable(resultado, claves):
    copia = resultado.copy()
    for nombre in ['anio', 'mes_num', 'reportes']:
        copia[nombre] = pd.to_numeric(copia[nombre], errors='raise').astype('Int64')
    for nombre in set(claves) - {'anio', 'mes_num'}:
        copia[nombre] = copia[nombre].astype('string')
    return copia.sort_values(claves, na_position='last').reset_index(drop=True)[claves + ['reportes']]

for nombre, claves in CONSULTAS.items():
    inicio = perf_counter()
    resultado = consultas_polars[nombre].collect(engine='streaming').to_pandas()
    primeras.append({'consulta': nombre, 'motor': 'polars_cpu',
                     'segundos': perf_counter() - inicio})
    pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
    print(nombre, 'pandas y Polars CPU coinciden grupo por grupo')


## L8. Comprobar NVIDIA y ejecutar Polars GPU
Seleccionar el kernel `.venv-gpu` antes de esta celda. `nvidia-smi` enumera las
NVIDIA del controlador, no la Intel ni necesariamente los dispositivos visibles
para CUDA si se restringió `CUDA_VISIBLE_DEVICES`. El índice siguiente es CUDA.

`raise_on_fail=True` impide que un plan no soportado termine silenciosamente
en CPU. **Si falla, no continuar al benchmark:** resolver el entorno o soporte
de la consulta. No se registra una comparación GPU sin haberla ejecutado.
`collect` devuelve el resultado al host; esa devolución y `.to_pandas()` están
incluidas en la frontera medida. La primera ejecución se registra aparte.


In [ ]:
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi is None and Path('/usr/lib/wsl/lib/nvidia-smi').is_file():
    nvidia_smi = '/usr/lib/wsl/lib/nvidia-smi'
assert nvidia_smi is not None, 'NVIDIA no comprobada: revisar controlador/WSL'
gpu_info = subprocess.run([nvidia_smi, '--query-gpu=index,name,uuid,memory.total,driver_version',
    '--format=csv'], capture_output=True, text=True, check=True).stdout
print(gpu_info)
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES', 'sin restricción explícita'))
engine_gpu = pl.GPUEngine(device=DISPOSITIVO_CUDA, raise_on_fail=True)
gpu_verificada = False
for nombre, claves in CONSULTAS.items():
    inicio = perf_counter()
    resultado = consultas_polars[nombre].collect(engine=engine_gpu).to_pandas()
    primeras.append({'consulta': nombre, 'motor': 'polars_gpu',
                     'segundos': perf_counter() - inicio})
    pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
    print(nombre, 'GPU ejecutada y resultado equivalente')
gpu_verificada = True


## L9. Benchmark: calentamiento y tres repeticiones
Las primeras ejecuciones anteriores se guardan aparte. Las siguientes celdas
hacen calentamiento adicional y luego rotan el orden de los tres motores.
La caché del sistema operativo puede persistir; esto **no es una prueba de disco
frío**. La creación de planes y las comprobaciones quedan fuera del reloj.
No se exige el mismo número de hilos; se registran hilos y hardware para
interpretar la configuración real. RAM pico y VRAM pico no se miden aquí.

La GPU también puede ser más lenta: queremos una conclusión medida, no una
aceleración garantizada. Descargar 1,82 GB no significa que cada consulta
transfiera 1,82 GB: el plan selecciona solo las columnas necesarias.


In [ ]:
assert gpu_verificada, 'Primero ejecutar correctamente L8'
def ejecutar(motor, nombre):
    if motor == 'pandas_cpu':
        return analizar_pandas(CONSULTAS[nombre])
    engine = 'streaming' if motor == 'polars_cpu' else engine_gpu
    return consultas_polars[nombre].collect(engine=engine).to_pandas()

motores = ['pandas_cpu', 'polars_cpu', 'polars_gpu']
for nombre, claves in CONSULTAS.items():
    for motor in motores:
        calentamiento = ejecutar(motor, nombre)
        pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(calentamiento, claves))
        del calentamiento

mediciones = []
for nombre, claves in CONSULTAS.items():
    for repeticion in range(REPETICIONES):
        desplazamiento = repeticion % len(motores)
        orden = motores[desplazamiento:] + motores[:desplazamiento]
        for motor in orden:
            inicio = perf_counter()
            resultado = ejecutar(motor, nombre)
            segundos = perf_counter() - inicio
            pd.testing.assert_frame_equal(comparable(referencias[nombre], claves), comparable(resultado, claves))
            assert int(resultado['reportes'].sum()) == TOTAL_ESPERADO
            mediciones.append({'consulta': nombre, 'motor': motor,
                'repeticion': repeticion + 1, 'segundos': segundos,
                'filas_entrada': TOTAL_ESPERADO, 'grupos': len(resultado), 'equivalente': True})
            print(nombre, repeticion + 1, motor, round(segundos, 4), 's', flush=True)
            del resultado


## L10. Analizar resultados, guardar evidencia y concluir
La distribución por corte permite comprobar el volumen de cada semestre.
Un ranking por entidad mide **reportes**, no tasas de riesgo: faltan denominadores
como población afiliada y exposición. Comparar cortes y cobertura antes de
interpretar cambios. Entidades con cadenas diferentes no se unifican automáticamente.

Factor de aceleración = mediana del motor CPU / mediana de Polars GPU.
Mayor que 1 favorece GPU; menor que 1 favorece ese motor CPU. Mantener separadas
las dos consultas y el tiempo de conversión. El cociente de tamaños CSV/Parquet
es compresión, no aceleración de procesamiento.


In [ ]:
for nombre, resultado in referencias.items():
    resultado.to_csv(OUT / f'{nombre}_verificado.csv', index=False)
por_corte = referencias['territorio_mes'].groupby('corte_archivo')['reportes'].sum()
assert por_corte.to_dict() == {f['periodo']: f['filas'] for f in FUENTES}
display(por_corte)
top_entidades = (referencias['territorio_mes_entidad']
    .groupby(['corte_archivo', 'ent_nombre'], dropna=False, as_index=False)['reportes'].sum()
    .sort_values(['corte_archivo', 'reportes', 'ent_nombre'], ascending=[True, False, True])
    .groupby('corte_archivo', sort=False).head(10))
display(top_entidades)
top_entidades.to_csv(OUT / 'top10_entidades_por_corte.csv', index=False)

tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == len(CONSULTAS) * len(motores) * REPETICIONES
resumen = tiempos.groupby(['consulta', 'motor'], as_index=False).agg(
    mediana_s=('segundos', 'median'), min_s=('segundos', 'min'), max_s=('segundos', 'max'))
comparacion = resumen.pivot(index='consulta', columns='motor', values='mediana_s')
comparacion['pandas_sobre_gpu'] = comparacion['pandas_cpu'] / comparacion['polars_gpu']
comparacion['polars_cpu_sobre_gpu'] = comparacion['polars_cpu'] / comparacion['polars_gpu']
display(resumen)
display(comparacion)

versiones = {}
for paquete in ['pandas', 'polars', 'pyarrow', 'cudf-polars-cu12', 'cudf-polars-cu13']:
    try:
        versiones[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError:
        versiones[paquete] = None
control_lab = {'periodos': PERIODOS, 'reportes': TOTAL_ESPERADO,
    'descargas': descargas, 'conversiones': conversiones,
    'gpu_ejecutada_y_equivalente': gpu_verificada,
    'dispositivo_cuda': DISPOSITIVO_CUDA, 'nvidia_smi': gpu_info,
    'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'versiones': versiones, 'python': sys.version, 'plataforma': platform.platform(),
    'cpu_logicas': os.cpu_count(), 'hilos_polars': pl.thread_pool_size(),
    'bloque_pandas': BLOQUE, 'repeticiones': REPETICIONES,
    'frontera': 'Parquet → agregación → resultado pandas en CPU',
    'descarga_y_conversion_incluidas_en_benchmark': False,
    'ram_pico_bytes': None, 'vram_pico_bytes': None}
tiempos.to_csv(OUT / 'benchmark_pandas_polars_gpu.csv', index=False)
pd.DataFrame(primeras).to_csv(OUT / 'primeras_ejecuciones.csv', index=False)
resumen.to_csv(OUT / 'resumen_benchmark.csv', index=False)
comparacion.to_csv(OUT / 'comparacion_medianas_y_factores.csv')
(OUT / 'control_laboratorio.json').write_text(json.dumps(control_lab, ensure_ascii=False, indent=2))
print('Evidencias:', OUT)


## L11. Trabajo que deben explicar los estudiantes
1. Mostrar el control de **tres CSV completos y tres Parquet**, sus filas y huellas.
2. Explicar separadores, códigos como texto, columnas derivadas y conservación de vacíos.
3. Dibujar cómo pandas combina conteos parciales y cómo Polars ejecuta el plan.
4. Demostrar equivalencia **grupo por grupo**, no solamente del total.
5. Comparar primera ejecución, medianas, dispersión y factores para las dos consultas.
6. Explicar lectura, transferencias host–GPU y temporales que pueden limitar la mejora.
7. ¿Qué ocurrió al añadir entidad a las claves? ¿Cambió el número de grupos y el rendimiento?
8. ¿Qué se reutilizó al reejecutar el laboratorio? ¿Qué quedó fuera del benchmark?

Integrar CSV, JSON e interpretación en el informe único de P04. No mezclar estos
tiempos con los de la primera parte o del CLI: sus fronteras son distintas.
Si no se pudo ejecutar GPU, informar el bloqueo y **no presentar factores GPU**.

Fuentes y preparación: [manifiesto PQRS](../pqrs_fuentes.json),
[P04](../../talleres/P04.md), [entorno GPU](../../docs/polars-gpu.md),
[motor GPU de Polars](https://docs.pola.rs/user-guide/gpu-support/).
